### Problem Statement: Bank Customer Churn Prediction

A bank is facing a problem where **many customers are leaving the bank**, but the management is unable to identify the reasons behind it.

The bank has approached us with its **customer data** to analyze the problem and understand why customers are leaving.


In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle


In [34]:
data=pd.read_csv('Dataset/Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [35]:
list_colomns=data.columns.tolist()
print(list_colomns)

['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']


In [36]:
data=data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


### Encoding Categorical Features

Our dataset contains two categorical columns, **Gender** and **Geography**, which have non-numerical values.

Since machine learning models require numerical inputs, we need to convert these categorical values into numerical representations.

1. **Gender — Label Encoding**
   - Contains only 2 categories: `Male` and `Female`.
   - We can use **Label Encoding** to convert them into numerical values.
   - Example: `Male → 1`, `Female → 0`.

2. **Geography — One-Hot Encoding**
   - Contains 3 categories: `France`, `Germany`, and `Spain`.
   - We use **One-Hot Encoding** to create separate binary columns for each category.
   - This avoids introducing an artificial order between countries.

**Key Point:** Label Encoding is suitable for binary categories in this case, while One-Hot Encoding is preferred for nominal categories with multiple values.

In [ ]:

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [38]:
one_hot_encoder_geography = OneHotEncoder(sparse_output=False, drop='first')
one_hot_encoded_data=one_hot_encoder_geography.fit_transform(data[['Geography']])
one_hot_encoded_data

array([[0., 0.],
       [0., 1.],
       [0., 0.],
       ...,
       [0., 0.],
       [1., 0.],
       [0., 0.]], shape=(10000, 2))

In [39]:
one_hot_encoder_geography.get_feature_names_out(['Geography'])

array(['Geography_Germany', 'Geography_Spain'], dtype=object)

In [40]:
one_hot_endoded_df = pd.DataFrame(one_hot_encoded_data, columns=one_hot_encoder_geography.get_feature_names_out(['Geography']))
one_hot_endoded_df.head()

,Geography_Germany,Geography_Spain
0,0.0,0.0
1,0.0,1.0
2,0.0,0.0
3,0.0,0.0
4,0.0,1.0


In [41]:
# Now we will drop the Geography coloumn from the original data and add the one hot encoded coloumns to the original data.
data = data.drop(['Geography'], axis=1)
data = pd.concat([data, one_hot_endoded_df], axis=1)    
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,1.0


In [42]:
# add those to pickle file so that we can use them later for prediction.
with open('label_encoder_gender.pkl', 'wb') as f:
    pickle.dump(label_encoder_gender, f)

with open('one_hot_encoder_geography.pkl', 'wb') as f:
    pickle.dump(one_hot_encoder_geography, f)

### Dividing the Dataset into Independent and Dependent Features

Before splitting the dataset into training, validation, and testing sets, we need to separate the **independent features (X)** and the **dependent feature (y)**.

1. **Independent Features (X):** Input features used by the model to make predictions, such as `CreditScore`, `Age`, `Balance`, and `Geography`.
2. **Dependent Feature (y):** The target variable that the model needs to predict. In our bank churn dataset, this is the `Exited` column.

**Example:**

```python
X = data.drop('Exited', axis=1)  # Independent features
y = data['Exited']               # Dependent feature
```

**Key Point:** The model learns the relationship between independent features (X) and the dependent feature (y) to predict whether a customer will leave the bank.

In [45]:
X = data.drop('Exited', axis=1)  # Independent features
y = data['Exited']               # Dependent feature

print(X.head())
print(y.head())

   CreditScore  Gender  ...  Geography_Germany  Geography_Spain
0          619       0  ...                0.0              0.0
1          608       0  ...                0.0              1.0
2          502       0  ...                0.0              0.0
3          699       0  ...                0.0              0.0
4          850       0  ...                0.0              1.0

[5 rows x 11 columns]
0    1
1    0
2    1
3    0
4    0
Name: Exited, dtype: int64


### Splitting the Dataset into Training, Validation, and Testing Sets

Now that we have converted all categorical columns into **numerical values**, we can split the dataset into three sets:

1. **Training Data (64%)** — Used to train the model and update its weights.
2. **Validation Data (16%)** — Used to monitor model performance, tune hyperparameters, and detect overfitting.
3. **Testing Data (20%)** — Used to evaluate the final model on unseen data.

**Splitting Strategy:**

- First, split the dataset into **80% training + validation** and **20% testing**.
- Then, split the 80% portion into **80% training** and **20% validation**.

**Final Dataset Distribution:**

```text
Complete Dataset (100%)
        |
        |--- Training (64%)
        |
        |--- Validation (16%)
        |
        |--- Testing (20%)
```

**Key Point:** Training data is used for learning, validation data is used for model tuning, and testing data is used for final performance evaluation.

In [56]:
# for this we only split the data into training and testing data not validation Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) 

### Feature Scaling of Independent Features

After separating the dataset into **independent features (X)** and **dependent feature (y)** and splitting the data into training, validation, and testing sets, we need to perform **Feature Scaling**.

#### Why is Feature Scaling Important?

- Different features may have different numerical ranges.
- Features with larger values can disproportionately influence model training.
- Feature scaling helps neural networks train more efficiently and converge faster.
- It helps improve the stability of gradient descent.

#### Standardization (StandardScaler)

We use `StandardScaler` to standardize the independent features.

After standardization, each feature in the training dataset has approximately:

- **Mean = 0**
- **Standard Deviation = 1**

**Formula:**

$$
z = \frac{x - \mu}{\sigma}
$$

Where:

- **x** = Original feature value
- **μ** = Mean of the feature
- **σ** = Standard deviation of the feature
- **z** = Standardized feature value


#### Why Do We Use `fit_transform()` Only on Training Data?

- **`fit_transform()`** calculates the mean and standard deviation from the training data and scales it.
- **`transform()`** uses the same mean and standard deviation calculated from the training data to scale validation and testing data.
- This prevents **data leakage**, ensuring the model does not gain information from the validation or testing datasets during preprocessing.

**Key Point:** Always fit the scaler on the training data only and use the same scaler to transform validation and testing data.

In [57]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((8000, 11), (2000, 11), (8000,), (2000,))

In [54]:
Scale=StandardScaler()
X_train=Scale.fit_transform(X_train)
X_test=Scale.transform(X_test)

In [55]:
#adding the scaler to pickle file so that we can use it later for prediction.
with open('scaler.pkl', 'wb') as f:
    pickle.dump(Scale, f)

# Important Concepts Before Building an ANN Using TensorFlow

## 1. Sequential Model

In TensorFlow, `Sequential()` is used to create a neural network where layers are connected **one after another**.

```python
from tensorflow.keras.models import Sequential

model = Sequential()
```

- Each layer passes its output to the next layer.
- Suitable for simple feedforward ANNs.

## 2. Dense Layer

In TensorFlow, `Dense()` creates a **fully connected layer**, where every neuron connects to all neurons in the previous layer.

```python
from tensorflow.keras.layers import Dense

model.add(Dense(64, activation='relu'))
```

- `64` → Number of neurons.
- `activation='relu'` → Activation function.
- Each neuron learns weights and a bias.

**Formula:**

$$
z = Wx + b
$$

## 3. Activation Functions

Activation functions introduce **non-linearity**, allowing neural networks to learn complex patterns.

| Activation | Output Range | Usage |
|---|---|---|
| `relu` | 0 to ∞ | Commonly used in hidden layers |
| `sigmoid` | 0 to 1 | Binary classification output |
| `tanh` | -1 to 1 | Hidden layers, some RNNs |
| `LeakyReLU` | Negative to positive | Helps prevent dying ReLU |

**TensorFlow Example:**

```python
model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
```

For bank churn prediction, **sigmoid** outputs the probability that a customer will leave.

## 4. Optimizer and Backpropagation

**Backpropagation** calculates gradients of the loss with respect to model weights.

The **optimizer** uses these gradients to update weights and reduce prediction errors.

**Weight Update Formula:**

$$
W_{new} = W_{old} - \eta \frac{\partial L}{\partial W}
$$

**TensorFlow Example:**

```python
model.compile(optimizer='adam')
```

Common optimizers:

- `SGD` → Basic gradient-based optimization.
- `Adam` → Adaptive learning rates; commonly used for ANNs.
- `RMSprop` → Adjusts learning rates using squared gradients.

**Key Point:** TensorFlow automatically performs backpropagation during `model.fit()`.

## 5. Loss Function

A loss function measures **how far the predicted output is from the actual output**.

In TensorFlow, we specify it using `model.compile()`.

```python
model.compile(
    optimizer='adam',
    loss='binary_crossentropy'
)
```

Common loss functions:

| Loss Function | Usage |
|---|---|
| `binary_crossentropy` | Binary classification |
| `categorical_crossentropy` | Multiclass classification with one-hot labels |
| `sparse_categorical_crossentropy` | Multiclass classification with integer labels |
| `mse` | Regression |

For bank churn prediction, we use **binary cross-entropy** because the output is either `0` (stays) or `1` (leaves).

## 6. Evaluation Metrics

Metrics measure **how well our neural network performs**.

In TensorFlow, metrics are defined inside `model.compile()`.

```python
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
```

Common metrics:

- `accuracy` → Percentage of correct predictions.
- `Precision` → Of customers predicted to leave, how many actually leave?
- `Recall` → Of customers who actually leave, how many did we identify?
- `AUC` → Measures the model's ability to distinguish between classes.
- `MAE` / `MSE` → Mainly used for regression.

**Key Point:** Accuracy alone may be misleading if the churn dataset is imbalanced.

## 7. Training Logs and TensorBoard

TensorFlow records training information such as **loss and accuracy across epochs**.

`TensorBoard` is used to visualize these logs.

```python
from tensorflow.keras.callbacks import TensorBoard

tensorboard = TensorBoard(log_dir='logs')
```

Pass the callback during training:

```python
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[tensorboard]
)
```

- `epochs=50` → Model goes through the entire training dataset 50 times.
- `batch_size=32` → Processes 32 training samples per batch.
- `validation_data` → Evaluates the model on validation data after each epoch.
- `callbacks` → Enables TensorBoard logging.

To launch TensorBoard in a terminal:

```bash
tensorboard --logdir logs
```

TensorBoard helps visualize training loss, validation loss, accuracy, and potential overfitting.

**Overall Flow:**

`Sequential → Dense Layers → Activation → Prediction → Loss → Backpropagation → Optimizer → Updated Weights → Evaluation`

In [63]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

### Building ANN Model

In [64]:
Model=Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)), # hidden layer which is connected to input layer, due to which we need to specify input shape
    Dense(32, activation='relu'), # 2nd hidden layer
    Dense(1, activation='sigmoid') #output layer, since we are doing binary classification we will use sigmoid activation function
])

d:\Udemy Ai-ML(Krishna Nayar)\New folder\3_ANN_End_to_End_Project\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [65]:
Model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,881 (11.25 KB)

 Trainable params: 2,881 (11.25 KB)

 Non-trainable params: 0 (0.00 B)

##### Compiling the model 

In [67]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
loss = tf.keras.losses.BinaryCrossentropy()


In [70]:
Model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])

#### Setting Up TensorBoard for Log Capture

**TensorBoard** is used to capture and visualize logs during neural network training.

### Implementation

```python
import datetime
from tensorflow.keras.callbacks import TensorBoard

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorboard_callback = TensorBoard(
    log_dir=log_dir,
    histogram_freq=1
)
```

### Parameter Explanation

- **`log_dir`** → Specifies the directory where training logs will be stored.
- **`datetime.datetime.now()`** → Gets the current date and time.
- **`strftime("%Y%m%d-%H%M%S")`** → Generates a unique timestamp for each training run.
- **`histogram_freq=1`** → Records weight and bias histograms after every epoch.

**Key Point:** TensorBoard stores logs for each training run in a separate timestamped directory, making it easier to analyze and compare runs.

In [83]:
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

#### Setting Up Early Stopping in TensorFlow

**Early Stopping** automatically stops model training when the monitored metric stops improving, helping **prevent overfitting and save training time**.

**Example:** If `epochs=1000` but validation loss stops improving at epoch 50, training can stop around epoch 60 when `patience=10`.

### 1. Implementation

```python
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    mode='min'
)
```

### 2. Understanding Parameters

| Parameter | Explanation |
|---|---|
| `monitor='val_loss'` | Monitors validation loss to determine improvement. Can also use `val_accuracy`. |
| `patience=10` | Stops training after 10 consecutive epochs without improvement. |
| `restore_best_weights=True` | Restores weights from the epoch with the best monitored value. |
| `mode='min'` | Lower values are better (`val_loss`). Use `max` for metrics like `val_accuracy`. |

### 3. Using Early Stopping During Training

```python
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=1000,
    batch_size=32,
    callbacks=[early_stopping]
)
```

- `epochs=1000` → Maximum number of epochs.
- `batch_size=32` → Processes 32 samples per batch.
- `validation_data` → Calculates validation metrics after each epoch.
- `callbacks` → Activates Early Stopping.

### 4. Example

| Epoch | Validation Loss | Status |
|---|---|---|
| 50 | 0.30 | Best performance |
| 51–59 | Above 0.30 | No improvement |
| 60 | 0.34 | Training stops |

**Result:** Training stops at epoch 60, and the weights from epoch 50 are restored.

**Key Point:** Early Stopping monitors the specified metric (usually `val_loss`), not simply whether training has completed. `patience` controls how long it waits before stopping.

In [84]:
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

#### Training the ANN Model

We use **`Model.fit()`** in TensorFlow to train our neural network using the training dataset.

### Implementation

```python
history = Model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping_callback, tensorboard_callback]
)
```

### Parameter Explanation

- **`X_train`** → Independent features used for training.
- **`y_train`** → Actual target values used for training.
- **`validation_data`** → Dataset used to evaluate model performance after each epoch.
- **`epochs=100`** → Maximum number of times the model processes the entire training dataset.
- **`batch_size=32`** → Processes 32 training samples per batch before updating weights.
- **`early_stopping_callback`** → Stops training when the monitored metric stops improving.
- **`tensorboard_callback`** → Captures training logs for visualization.
- **`history`** → Stores training and validation metrics, such as loss and accuracy, for each epoch.

**Important:** Since we have a separate validation dataset, we should use:

```python
validation_data=(X_val, y_val)
```

This keeps the **test dataset untouched** until final model evaluation.

**Key Point:** `Model.fit()` trains the neural network using backpropagation and the configured optimizer, while callbacks monitor training progress.

In [85]:
history=Model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=100,batch_size=32,callbacks=[early_stopping_callback,tensorboard_callback])

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6871 - loss: 26.5600 - val_accuracy: 0.4715 - val_loss: 17.3604
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6846 - loss: 21.0244 - val_accuracy: 0.8025 - val_loss: 23.0246
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6921 - loss: 15.0055 - val_accuracy: 0.7245 - val_loss: 5.3658
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.6930 - loss: 15.2046 - val_accuracy: 0.5010 - val_loss: 18.3475
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6910 - loss: 17.4284 - val_accuracy: 0.2585 - val_loss: 34.3445
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.6898 - loss: 11.6859 - val_accuracy: 0.7950 - val_loss: 9.5808
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6971 - loss: 13.2271 - val_accuracy: 0.2190 - val_loss: 31.7777
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6886 - loss: 16.99

In [86]:
Model.save('model.h5')

In [87]:
# load tensorboard extension
%load_ext tensorboard
%tensorboard --logdir logs/fit

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 24572), started 0:05:26 ago. (Use '!kill 24572' to kill it.)